# CineMidas RAG

Notebook de desenvolvimento do agente CineMidas, criado para responder dúvidas frequentes com base no Manual de Atendimento da Rede CineViva.

## Objetivo desta etapa

Preparar e validar o ambiente de desenvolvimento no Google Colab.

## Etapas planejadas

1. Configurar as dependências.
2. Carregar o manual em PDF.
3. Extrair e dividir o texto.
4. Criar representações semânticas.
5. Recuperar os trechos relevantes.
6. Gerar respostas com o Gemini.
7. Avaliar as respostas.
8. Transferir a aplicação validada para uma interface Streamlit.

In [1]:
import sys

print(f"Python: {sys.version}")

Python: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]


In [2]:
from google.colab import userdata

gemini_key = userdata.get("GEMINI_API_KEY")

print("GEMINI_API_KEY configurada:", bool(gemini_key))

GEMINI_API_KEY configurada: True


## 1. Instalação das dependências

In [3]:
%pip install -qU \
    "requests==2.32.4" \
    langchain \
    langchain-google-genai \
    langchain-text-splitters \
    pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.6/139.6 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.2/72.2 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.5/349.5 kB 15.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 558.5/558.5 kB 25.8 MB/s eta 0:00:00


In [4]:
from pypdf import PdfReader
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_google_genai import (
    ChatGoogleGenerativeAI,
    GoogleGenerativeAIEmbeddings,
)
from langchain_core.vectorstores import InMemoryVectorStore

print("Dependências importadas com sucesso.")

Dependências importadas com sucesso.


In [5]:
%pip check

ipython 7.34.0 requires jedi, which is not installed.


## 2. Carregamento do documento

In [6]:
from pathlib import Path
import requests

PDF_URL = (
    "https://raw.githubusercontent.com/"
    "takatonto/cinemidas-rag/main/"
    "documents/manual_atendimento_cineviva.pdf"
)

PDF_PATH = Path("/content/manual_atendimento_cineviva.pdf")

response = requests.get(PDF_URL, timeout=30)
response.raise_for_status()

if not response.content.startswith(b"%PDF"):
    raise ValueError("O arquivo baixado não é um PDF válido.")

PDF_PATH.write_bytes(response.content)

print(f"PDF carregado: {PDF_PATH.name}")
print(f"Tamanho: {PDF_PATH.stat().st_size / 1024:.1f} KB")

PDF carregado: manual_atendimento_cineviva.pdf
Tamanho: 41.6 KB


## 3. Extração do texto do PDF

In [7]:
reader = PdfReader(str(PDF_PATH))

documents = []

for page_number, page in enumerate(reader.pages, start=1):
    text = (page.extract_text() or "").strip()

    if text:
        document = Document(
            page_content=text,
            metadata={
                "source": PDF_PATH.name,
                "page": page_number,
                "document_id": "CV-MAN-ATD-001",
            },
        )
        documents.append(document)

if not documents:
    raise ValueError("Nenhum texto foi extraído do PDF.")

total_characters = sum(len(document.page_content) for document in documents)

print(f"Páginas do PDF: {len(reader.pages)}")
print(f"Páginas com texto: {len(documents)}")
print(f"Caracteres extraídos: {total_characters}")

Páginas do PDF: 10
Páginas com texto: 10
Caracteres extraídos: 22234


In [8]:
print(documents[0].page_content[:1000])
print("\nMetadados:", documents[0].metadata)

---title: Manual de Atendimento ao Cliente — Rede CineVivadocument_id: CV-MAN-ATD-001version: "1.0"last_updated: "2026-07-23"language: "pt-BR"status: "Documento fictício para projeto educacional"---
# Manual de Atendimento ao Cliente — Rede CineViva
## 1. Sobre este documento
Este manual reúne as principais políticas e orientações de atendimento da Rede CineViva, uma empresa fictícia do setor de exibição cinematográfica.
O documento deve ser utilizado pelos colaboradores da CineViva para esclarecer dúvidas frequentes dos clientes sobre ingressos, pagamentos, cancelamentos, sessões, acessibilidade, alimentos, programa de fidelidade e atendimento.
As regras deste documento são fictícias e foram criadas exclusivamente para um projeto educacional de inteligência artificial. Elas não representam as políticas de uma empresa real.
## 2. Escopo do assistente CineMidas
O CineMidas é o assistente virtual interno da Rede CineViva.
O CineMidas pode:
- Responder perguntas com base neste manual.- Ex